# 07 · Export a dome as a mesh file (OBJ, OFF, NumPy .npz)

The exported mesh is de-duplicated (every physical point once) and every triangle is wound
counter-clockwise seen from outside, so normals point outwards. Open the `.obj` / `.off` in Blender,
MeshLab, three.js, trimesh, etc. Files are written to `examples/output/` (they are git-ignored).

Script version: [`examples/07_export_mesh.py`](../07_export_mesh.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dome_utils import output_path, shade, unique_mesh
from ipywidgets import Checkbox, FloatSlider, IntSlider, interact_manual
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from mt.geodesicdome.grid.geodesicdome import GeodesicDome

## Build and check the mesh

In [ ]:
FREQ = 6
RADIUS = 1.0

dome = GeodesicDome(FREQ)
points, faces, _ = unique_mesh(dome)
points = points * RADIUS

# sanity check: the Euler characteristic of a sphere is V - E + F = 2
edges = {tuple(sorted(e)) for f in faces for e in ((f[0], f[1]), (f[1], f[2]), (f[2], f[0]))}
print(f'V={len(points)}  E={len(edges)}  F={len(faces)}  V-E+F={len(points) - len(edges) + len(faces)}')

# outward winding: the normal of every face points away from the centre
tri = points[faces]
normals = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
print('all normals point outwards?', bool(np.all((normals * tri.mean(axis=1)).sum(axis=1) > 0)))

## Write OBJ, OFF and NPZ

In [ ]:
def export(points, faces, base, obj=True, off=True, npz=True, comment=''):
    """Writes the mesh to <base>.obj / .off / .npz and returns the paths written."""
    edges = {tuple(sorted(e)) for f in faces for e in ((f[0], f[1]), (f[1], f[2]), (f[2], f[0]))}
    written = []
    if obj:
        with open(base + '.obj', 'w') as fh:                   # OBJ: 1-based indices
            fh.write(f'# {comment}\n')
            for x, y, z in points:
                fh.write(f'v {x:.9f} {y:.9f} {z:.9f}\n')
            for a, b, c in faces + 1:
                fh.write(f'f {a} {b} {c}\n')
        written.append(base + '.obj')
    if off:
        with open(base + '.off', 'w') as fh:                   # OFF: 0-based indices
            fh.write(f'OFF\n{len(points)} {len(faces)} {len(edges)}\n')
            for x, y, z in points:
                fh.write(f'{x:.9f} {y:.9f} {z:.9f}\n')
            for a, b, c in faces:
                fh.write(f'3 {a} {b} {c}\n')
        written.append(base + '.off')
    if npz:
        np.savez(base + '.npz', points=points, faces=faces)    # reload with np.load(...)
        written.append(base + '.npz')
    return written


base = output_path(f'geodesic_f{FREQ}')
for path in export(points, faces, base, comment=f'geodesic dome, frequency {FREQ}, radius {RADIUS}'):
    print('saved', path)

The first lines of the OBJ file:

In [ ]:
with open(base + '.obj') as fh:
    for _, line in zip(range(6), fh, strict=False):
        print(line, end='')

## Reload the `.npz` and look at it

In [ ]:
def preview(points, faces, title):
    tri = points[faces]
    normals = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    normals /= np.linalg.norm(normals, axis=1, keepdims=True)
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(projection='3d')
    ax.add_collection3d(Poly3DCollection(tri, facecolors=shade(normals)[:, None] * np.array([0.27, 0.51, 0.71]),
                                         edgecolors='white', linewidths=0.3))
    r = np.abs(points).max()
    ax.set_xlim(-r, r), ax.set_ylim(-r, r), ax.set_zlim(-r, r)
    ax.set_box_aspect((1, 1, 1)), ax.set_axis_off(), ax.set_title(title)
    plt.show()


mesh = np.load(base + '.npz')
assert np.allclose(mesh['points'], points) and np.array_equal(mesh['faces'], faces)
preview(mesh['points'], mesh['faces'], f"reloaded: {len(mesh['points'])} points, {len(mesh['faces'])} faces")

### Try it: export your own

Pick a frequency, a radius and the formats, then press **Run Interact**.

In [ ]:
@interact_manual(f=IntSlider(value=6, min=1, max=64, description='frequency'),
                 radius=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='radius'),
                 obj=Checkbox(value=True, description='.obj'),
                 off=Checkbox(value=True, description='.off'),
                 npz=Checkbox(value=True, description='.npz'),
                 show=Checkbox(value=True, description='preview'))
def export_dome(f, radius, obj, off, npz, show):
    p, fc, _ = unique_mesh(GeodesicDome(f))
    p = p * radius
    for path in export(p, fc, output_path(f'geodesic_f{f}'), obj, off, npz,
                       comment=f'geodesic dome, frequency {f}, radius {radius}'):
        print('saved', path)
    if show:
        preview(p, fc, f'frequency {f}, radius {radius}: {len(p)} points, {len(fc)} faces')